In [214]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import sys
import json
from datetime import datetime
import numpy as np


In [215]:
from typing import get_origin, get_args, Literal

In [216]:
MODEL = "qwen3:4b"

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)


In [217]:
class Memory:
    def __init__(self, embedding_function, file_path = "memory.json"):
        self.data = []
        self.next_id = 1
        self.embedding_function = embedding_function
        self.file_path = file_path

        self.load_from_disk()

    def save_to_disk(self):
        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump(self.data, f, indent=4)

    def load_from_disk(self):
        if not os.path.exists(self.file_path):
            return f"File path does not exist!!"

        with open(self.file_path, "r", encoding = "utf-8") as f:
            self.data = json.load(f)

        if self.data:
            self.next_id = max(memory["id"] for memory in self.data) + 1
            

    def remember(self, key, value, category = "other"):
        text = f"{key} : {value}"
        embedding = self.embedding_function(text)

        for memory in self.data:

            if memory["key"] == key:
                memory["value"] = value
                memory["text"] = text
                memory["embedding"] = embedding
                memory["category"] = category
                memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                self.save_to_disk()

                return

        self.data.append({
            "id" : self.next_id,
            "key": key,
            "value": value,
            "text" : f"{key} : {value}",
            "embedding": embedding,
            "category": category,
            "timestamp" : datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        
        self.next_id += 1
        self.save_to_disk()
    
    def recall(self, key):
        for memory in reversed(self.data):
            if memory["key"] == key:
                return memory["value"]

    def forget(self, key):
        original_count = len(self.data)

        self.data = [
            memory
            for memory in self.data
            if memory["key"] != key
        ]

        if len(self.data) < original_count:
            self.save_to_disk()
            return True

        return False

    def list_memories(self):
        return self.data

    def search(self, query, top_k = 3, threshold = 0.55, category = None):
        query_embedding = self.embedding_function(query)

        results = []

        for memory in self.data:

            if category is not None and memory.get("category") != category:
                continue

            score = cosine_similarity(query_embedding, memory["embedding"])

            if score >= threshold:
                results.append({
                    "memory" : memory,
                    "score" : score
                })

        results.sort(
            key = lambda x : x["score"],
            reverse = True
        )

        return results[:top_k]


In [218]:
class Tool:
    def __init__(self, function, description):
        self.function = function
        self.description = description
        self.parameters = generate_parameters(function)

    def execute(self, arguments, context):
        try:
            if context is None:
                context = {}
            
            return self.function(**arguments, **context)
        except Exception as e:
            return f"Tool execution failed: {str(e)}"

    def schema(self):
        return {
            "type" : "function",
            "function":{
                "name": self.function.__name__,
                "description": self.description,
                "parameters": self.parameters
            }
        }
    

In [219]:
class Agent:
    def __init__(self, client, tool_list, model, system_prompt):
        self.client = client
        self.model = model
        self.tool_list = tool_list

        self.TOOL_MAP = {
            tool.function.__name__ : tool
            for tool in tool_list
        }

        self.Tool_SCHEMA = [
            tool.schema()
            for tool in tool_list
        ]

        self.messages = [{
            "role": "system",
            "content": system_prompt
        }]

        self.state = {}
        self.memory = Memory(create_embedding, file_path = "memory.json")
        self.context = {"memory": self.memory}
        
    def set_state(self, key, value):
        self.state[key] = value

    def get_state(self, key):
        return self.state[key]

    def call_llm(self):
            return self.client.chat.completions.create(
                    model=self.model,
                    messages=self.messages,
                    tools=self.Tool_SCHEMA,
                    max_tokens=1000
                )

    def execute(self, tool_call):
            tool_name = tool_call.function.name
            print(f"TOOL CALLED: {tool_name}")
            tool = self.TOOL_MAP.get(tool_name)
            if not tool:
                return f"Tool '{tool_name}' does not exist."
            try:
                arguments = json.loads(tool_call.function.arguments)
                return tool.execute(arguments, self.context)
            except Exception as e:
                return f"Tool execution failed: {str(e)}"
        
    def run(self, user_input):
        self.messages.append({"role": "user", "content": user_input})
        MAX_ITERATIONS = 10
        for iteration in range(MAX_ITERATIONS):
            response = self.call_llm()
            response_message = response.choices[0].message
            self.messages.append(response_message)
            if not response_message.tool_calls:
                return f"AI: ",response_message.content
            for tool_call in response_message.tool_calls:
                    print("Tool is running")
                    result = self.execute(tool_call)
                    self.messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": json.dumps(result)
                    })
        else:
            print("MAX Tool iterations reached!!")


In [220]:
EMBEDDING_MODEL = "Qwen3-Embedding:4B"

embedding_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)


In [221]:
def create_embedding(text):
    embedding = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
        encoding_format="float"
    )
    return embedding.data[0].embedding


In [222]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [223]:
def python_type_to_json_type(annotation):

    if get_origin(annotation) is Literal:

        values = get_args(annotation)

        first_value = values[0]

        if isinstance(first_value, str):
            json_type = "string"
        
        elif isinstance(first_value, int):
            json_type = "integer"

        elif isinstance(first_value, float):
            json_type = "number"
        
        elif isinstance(first_value, bool):
            json_type = "boolean"

        else:
            json_type = "string"

        return{
            "type": json_type,
            "enum": list(values)
        }
    if annotation == str:
        return "string"

    elif annotation == int:
        return "integer"

    elif annotation == float:
        return "number"

    elif annotation == bool:
        return "boolean"

    return "string"

In [224]:
import inspect

def generate_parameters(function):
    
    signature = inspect.signature(function)

    properties = {}
    required = []

    for name, parameter in signature.parameters.items():
        if name == "memory":
            continue

        json_type = python_type_to_json_type(parameter.annotation)

        if isinstance(json_type, dict):
            properties[name] = json_type

        else:
            properties[name] = {
                "type" : json_type
            }

        if parameter.default is inspect.Parameter.empty:
            required.append(name)

    return {
        "type" : "object",
        "properties" : properties,
        "required" : required
    }

In [225]:
def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [226]:
time_tool = Tool(
    function = get_current_time,
    description="Get current Time",
)

In [227]:
def calculator(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]):
    if(operation == "add"):
        return a+b

    elif(operation == "divide"):
        if(b != 0):
            return a/b
        else: return "Cannot divide with Zero"

    elif(operation == "subtract"):
        return a-b
    
    elif(operation == "multiply"):
        return a*b


In [228]:
calculator_tool = Tool(
    function=calculator,
    description="Perform mathematical calculations",
)

In [229]:
def greet(name: str, age: int, excited: bool = False):
    if excited:
        return f"Hello {name}! You are {age} years old!"
    return f"Hello {name}. You are {age} years old."

In [230]:
greet_tool = Tool(
    greet,
    "Greets a Person"
)

In [231]:
def save_memory(memory: Memory, key: str, value: str, category: str = "other"):
    memory.remember(key, value, category)
    return f"Remembered {key} = {value}"

In [232]:
memory_tool = Tool(
    save_memory,
    "Saves important information to agent's memory"
)

In [233]:
def recall_memory(memory: Memory, key: str):
    value = memory.recall(key)
    if value is None:
        return f"No memory found for '{key}'"

    return f"The value of {key} = {value}"

In [234]:
recall_tool = Tool(
    recall_memory,
    """Retrieve a memory using its exact key.

    Use this ONLY when you already know the exact memory key.
    For example, if the key is exactly "hometown", use:
    recall_memory(key="hometown").

    If you do not know the exact key, use search_memory instead."""
)

In [235]:
def forget_memory(memory: Memory, key: str):
    deleted = memory.forget(key)
    if not deleted:
        return f"No memory found : {key}"
    return f"Memory deleted : {key}"

In [236]:
forget_tool = Tool(
    forget_memory,
    """Forget (delete) a memory by its exact key.
    
    IMPORTANT: You must know the EXACT key before calling this.
    If unsure, call search_memory first to find the correct key,
    then call this with the exact key returned."""
)

In [237]:
def get_all_memories(memory: Memory):
    return memory.list_memories()

In [238]:
def search_memory(memory: Memory, query: str, category: str = None):
    print("Using search memory")
    results =  memory.search(query, category = category)

    if not results:
        return {
            "found": False,
            "memories": []
            }

    cleaned_results = []

    for result in results:
        memory_data = result["memory"]

        cleaned_results.append({
            "key": memory_data["key"],
            "value": memory_data["value"],
            category: memory_data.get("category","other"),
            "score": round(result["score"], 3)
        })

    return {
        "found": True,
        "memories": cleaned_results
    }

In [239]:
search_memory_tool = Tool(
    search_memory,
    """Search the agent's memory using keywords when you are unsure of the
    exact memory key. Use this tool when the user asks about something
    that may be stored in memory but you do not know the exact key.

    Example:
    User asks "What programming language do I like?"
    Search using query="language".

    Do NOT use recall_memory unless you know the exact key."""
)

In [240]:
list_memory_tool = Tool(
    get_all_memories,
    "get all the memories currently stored by the agent"
)

In [241]:
tool_list = [
    calculator_tool,
    time_tool,
    greet_tool,
    memory_tool,
    recall_tool,
    forget_tool,
    search_memory_tool
]

In [242]:
agent = Agent(
    client=client,
    tool_list=tool_list,
    model=MODEL,
    system_prompt="""You are a helpful AI agent.

You have access to tools for calculations, getting the current time,
and managing memory.

Use the calculator for mathematical calculations.
Use the time tool when the user asks for the current time.

Memory rules:

1. When the user explicitly asks you to remember something,
   use save_memory.

3. If you do NOT know the exact memory key, use search_memory.
   Do not guess the key.

4. Never use get_all_memories.

5. Do not invent memories.

6. When the user asks you to forget something, ALWAYS use
   search_memory first to find the exact key, then call
   forget_memory with that exact key. Never guess the key.
   
7. When saving memories, classify them using these categories:

- identity: information that directly identifies the user, such as name, age, or occupation.
- preference: likes, dislikes, and favorites.
- personal: personal facts such as hometown, college, or background.
- context: temporary or current information such as current projects or goals.
- other: information that does not clearly fit the above categories.

8. When searching memory, use a category when the user's question clearly relates to a specific category.

Available categories:
- identity
- preference
- personal
- context
- other

If the question is broad or does not clearly belong to one category, search without specifying a category."""
)


In [243]:
while True:
    try:
        user_input = input("You: ")
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input.strip():
        continue

    if user_input.lower().strip() == "exit":
        break

    try:
        response = agent.run(user_input)
        print(response)
    except Exception as e:
        print("Error:", e)

Tool is running
TOOL CALLED: search_memory
Using search memory
('AI: ', 'Your name is sairamesh.')


In [244]:
print(agent.memory.search("What is my favorite food!"))

[]


In [245]:
agent.memory.list_memories()

[{'id': 1,
  'key': 'name',
  'value': 'sairamesh',
  'text': 'name : sairamesh',
  'embedding': [-0.00014031485,
   0.02398927,
   -0.013396178,
   0.03813322,
   -0.0004320904,
   0.052493002,
   0.061188728,
   0.019224143,
   0.025795948,
   0.026237,
   0.0075002476,
   -0.015885055,
   0.0013156849,
   -0.033507142,
   0.062456436,
   -0.016142026,
   0.0198972,
   -0.060807683,
   0.021838166,
   -0.0055959416,
   -0.013720768,
   -0.022364568,
   0.03946172,
   -0.009306858,
   0.00332442,
   0.016508661,
   -0.0258481,
   -0.01848344,
   0.018027462,
   0.035526697,
   -0.02727558,
   -0.045521908,
   0.02507617,
   0.027017973,
   0.0043919706,
   -0.03353081,
   0.0038447615,
   -0.01657845,
   -0.0062250886,
   -0.038781036,
   0.049202148,
   -0.013592989,
   -0.00057354127,
   -0.012548262,
   -0.011977656,
   -0.02936651,
   -0.0019889646,
   -0.006358344,
   0.021349458,
   -0.025966886,
   0.0044141687,
   0.0050724754,
   -0.006502937,
   -0.022625638,
   -0.004813226

In [246]:
print(agent.memory.search(
    "name?",
    category="personal"
))

[]


In [247]:
print(agent.memory.search(
    "what is my name?",
    category="identity"
))

[{'memory': {'id': 1, 'key': 'name', 'value': 'sairamesh', 'text': 'name : sairamesh', 'embedding': [-0.00014031485, 0.02398927, -0.013396178, 0.03813322, -0.0004320904, 0.052493002, 0.061188728, 0.019224143, 0.025795948, 0.026237, 0.0075002476, -0.015885055, 0.0013156849, -0.033507142, 0.062456436, -0.016142026, 0.0198972, -0.060807683, 0.021838166, -0.0055959416, -0.013720768, -0.022364568, 0.03946172, -0.009306858, 0.00332442, 0.016508661, -0.0258481, -0.01848344, 0.018027462, 0.035526697, -0.02727558, -0.045521908, 0.02507617, 0.027017973, 0.0043919706, -0.03353081, 0.0038447615, -0.01657845, -0.0062250886, -0.038781036, 0.049202148, -0.013592989, -0.00057354127, -0.012548262, -0.011977656, -0.02936651, -0.0019889646, -0.006358344, 0.021349458, -0.025966886, 0.0044141687, 0.0050724754, -0.006502937, -0.022625638, -0.0048132264, 0.0099984305, 0.0417735, -0.0112722665, -0.014857369, -0.010590661, 0.006213841, 0.018475285, -0.050928667, -0.025007406, -0.004229802, -0.03372929, -0.0133